In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler , LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error , r2_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.tree import DecisionTreeRegressor


C:\Users\vivek\anaconda4\lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
df = pd.read_csv("train_energy_data.csv")
df.head()

,Building Type,Square Footage,Number of Occupants,Appliances Used,Average Temperature,Day of Week,Energy Consumption
0,Residential,7063,76,10,29.84,Weekday,2713.95
1,Commercial,44372,66,45,16.72,Weekday,5744.99
2,Industrial,19255,37,17,14.30,Weekend,4101.24
3,Residential,13265,14,41,32.82,Weekday,3009.14
4,Commercial,13375,26,18,11.92,Weekday,3279.17


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Building Type        1000 non-null   object 
 1   Square Footage       1000 non-null   int64  
 2   Number of Occupants  1000 non-null   int64  
 3   Appliances Used      1000 non-null   int64  
 4   Average Temperature  1000 non-null   float64
 5   Day of Week          1000 non-null   object 
 6   Energy Consumption   1000 non-null   float64
dtypes: float64(2), int64(3), object(2)
memory usage: 54.8+ KB


In [4]:
categorical_cols = df.select_dtypes(include = 'object').columns
print(categorical_cols)

Index(['Building Type', 'Day of Week'], dtype='object')


In [5]:
X = df.drop("Energy Consumption", axis=1)
y = df["Energy Consumption"]

print(X.shape)
print(y.shape)

(1000, 6)
(1000,)


In [6]:
X_train , X_val, y_train , y_val = train_test_split(X, y ,test_size = 0.2 , random_state =42)

In [7]:
ohe = OneHotEncoder(drop = "first", sparse_output =False, handle_unknown='ignore' )

In [8]:
X_train_cat = ohe.fit_transform(X_train[categorical_cols])
X_val_cat = ohe.transform(X_val[categorical_cols])

In [9]:
encoded_cols = ohe.get_feature_names_out(categorical_cols)

X_train_cat_df = pd.DataFrame(X_train_cat , columns = encoded_cols ,index=X_train.index)
X_val_cat_df = pd.DataFrame(X_val_cat , columns = encoded_cols, index=X_val.index)


In [10]:
X_train_num = X_train.drop(columns=categorical_cols)
X_val_num = X_val.drop(columns=categorical_cols)

X_train_final = pd.concat([X_train_num, X_train_cat_df], axis=1)
X_val_final = pd.concat([X_val_num, X_val_cat_df], axis=1)
print(X_train_final.shape)
print(X_val_final.shape)


(800, 7)
(200, 7)


In [11]:
numerical_cols = ['Square Footage', 'Number of Occupants', 'Appliances Used', 'Average Temperature']
scaler = StandardScaler()

X_train_num_scaled = scaler.fit_transform(X_train_final[numerical_cols])
X_val_num_scaled = scaler.transform(X_val_final[numerical_cols])


X_train_num_scaled_df = pd.DataFrame(
    X_train_num_scaled,
    columns=numerical_cols,
    index=X_train_final.index
)

X_val_num_scaled_df = pd.DataFrame(
    X_val_num_scaled,
    columns=numerical_cols,
    index=X_val_final.index
)

# For training set
X_train_scaled = pd.concat(
    [X_train_num_scaled_df, X_train_final.drop(columns=numerical_cols)],
    axis=1
)

# For validation set
X_val_scaled = pd.concat(
    [X_val_num_scaled_df, X_val_final.drop(columns=numerical_cols)],
    axis=1
)
print(X_train_scaled.head())
print(X_val_scaled.head())


     Square Footage  Number of Occupants  Appliances Used  \
29         0.120184            -1.214780         0.321025   
535       -1.074908            -1.453382        -0.466767   
695       -0.202598            -1.351124        -0.896472   
557        0.332615            -0.635319         0.535878   
836       -0.624890            -1.146608         0.893966   

     Average Temperature  Building Type_Industrial  Building Type_Residential  \
29              1.443981                       0.0                        1.0   
535            -0.270429                       0.0                        0.0   
695             0.508339                       1.0                        0.0   
557             0.160975                       0.0                        1.0   
836            -0.586979                       0.0                        0.0   

     Day of Week_Weekend  
29                   1.0  
535                  0.0  
695                  0.0  
557                  1.0  
836        

In [12]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=300, max_depth=15, random_state=42),
    "XGBOOST": XGBRegressor(random_state=42)
}

# ----------------------------
# 2️⃣ Train models and store them
# ----------------------------
trained_models = {}   # Dictionary to store fitted models
results = {}          # Dictionary to store validation metrics

for name, model in models.items():
    print(f"Training {name}")
    
    # Fit model on training data
    model.fit(X_train_scaled, y_train)
    
    # Store the trained model
    trained_models[name] = model
    
    # Predict on validation set
    val_pred = model.predict(X_val_scaled)
    
    # Calculate metrics
    mse = mean_squared_error(y_val, val_pred)
    rmse = mean_squared_error(y_val, val_pred, squared=False)
    r2 = r2_score(y_val, val_pred)
    
    # Save metrics
    results[name] = {"MSE": mse, "RMSE": rmse, "R2": r2}


Training Linear Regression
Training Decision Tree
Training Random Forest


C:\Users\vivek\anaconda4\lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
C:\Users\vivek\anaconda4\lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
C:\Users\vivek\anaconda4\lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


Training XGBOOST


C:\Users\vivek\anaconda4\lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [13]:
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values(by='R2', ascending=False)
print("Validation Results:")
print(results_df)

Validation Results:
                            MSE        RMSE        R2
Linear Regression      0.000189    0.013731  1.000000
XGBOOST            12507.217513  111.835672  0.984641
Random Forest      15273.888474  123.587574  0.981243
Decision Tree      47609.053772  218.194990  0.941534


In [14]:
test_df = pd.read_csv("test_energy_data.csv")
X_test = test_df.drop(columns=["Energy Consumption"])

In [15]:
X_test_cat = ohe.transform(X_test[categorical_cols])
X_test_cat_df = pd.DataFrame(X_test_cat, columns=encoded_cols, index=X_test.index)

In [16]:
X_test_num = X_test.drop(columns=categorical_cols)
X_test_num_scaled = scaler.transform(X_test_num[numerical_cols])
X_test_num_scaled_df = pd.DataFrame(X_test_num_scaled, columns=numerical_cols, index=X_test.index)

In [17]:
X_test_final = pd.concat([X_test_num_scaled_df, X_test_cat_df], axis=1)

# Align test columns with training set
X_test_final = X_test_final[X_train_scaled.columns]

In [18]:
linear_model = trained_models["Linear Regression"]  # Use the trained model
y_test_pred = linear_model.predict(X_test_final)

In [19]:
y_test_true = test_df["Energy Consumption"]
test_mse = mean_squared_error(y_test_true, y_test_pred)
test_rmse = mean_squared_error(y_test_true, y_test_pred, squared=False)
test_r2 = r2_score(y_test_true, y_test_pred)

print("Test MSE:", test_mse)
print("Test RMSE:", test_rmse)
print("Test R2:", test_r2)

Test MSE: 0.0002007108907385126
Test RMSE: 0.014167247112213177
Test R2: 0.9999999997075142


C:\Users\vivek\anaconda4\lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
